# Consolidación HAI22.04: TEST1–TEST4 (residuos originales)

La detección se reconstruye a partir de las columnas `error__*` guardadas en las predicciones originales, evitando diferencias de redondeo al volver a restar `real__* - pred__*`. Se exige reproducción exacta de las alertas previas.


In [1]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(r'C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive\resultados_hai_arboles')
OUT = ROOT/'analisis'/'consolidado_4_test'
OUT.mkdir(parents=True, exist_ok=True)
T1 = ROOT/'analisis'/'test1'
TESTS = ('test1','test2','test3','test4')
METHODS = ('DT','RF','ET','XGB','PERSIST')
CONFIGS = ('FULL','SIN_P1','SIN_P2','SIN_P3','SIN_P4')
COMPS = ('P1','P2','P3','P4')
Q_VALUES = (0.990,0.995,0.999)
EXPECTED_N = {'test1':86370,'test2':82770,'test3':62370,'test4':129570}
EXPECTED_EPISODES = {'test1':7,'test2':17,'test3':10,'test4':24}
assert (ROOT/'predicciones').is_dir() and (ROOT/'deteccion').is_dir()
assert T1.is_dir(), 'Ejecutar primero 11_evaluacion_TEST1_HAI2204.ipynb'
print('Origen:',ROOT,'\nSalida:',OUT)

Origen: C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive\resultados_hai_arboles 
Salida: C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive\resultados_hai_arboles\analisis\consolidado_4_test


In [2]:
def load_csv(path, **kwargs):
    if not path.is_file(): raise FileNotFoundError(str(path))
    with path.open('rb') as f: gzip = f.read(2) == bytes((31,139))
    return pd.read_csv(path, compression='gzip' if gzip else None, **kwargs)

def locate(kind, method, config, test):
    stem = f'{method}_{config}_{test}'
    if test=='test1':
        direct=T1/(stem+'_'+('predicciones' if kind=='predicciones' else 'deteccion')+'.csv.gz')
        if direct.is_file():return direct
        places=(T1/kind, T1)
    else: places=(ROOT/kind,)
    for directory in places:
        for ext in ('.csv.gz','.csv'):
            q=directory/(stem+ext)
            if q.is_file(): return q
    raise FileNotFoundError(f'No existe {kind}/{stem} en {places}')

def get_threshold(method,config):
    p=ROOT/'deteccion'/f'umbral_{method}_{config}.json'
    if not p.exists():raise FileNotFoundError(str(p))
    j=json.loads(p.read_text(encoding='utf-8'))
    cols=list(j['variables']); mu=np.asarray(j['mu'],float); sd=np.asarray(j['std'],float)
    assert len(cols)==len(mu)==len(sd)==24 and np.all(sd>0)
    return cols,mu,sd,float(j['umbral'])

def runs(indices, flags):
    x=np.asarray(indices,dtype=np.int64); f=np.asarray(flags,bool)
    assert len(x)==len(f) and (len(x)<2 or np.all(np.diff(x)>0))
    s=np.flatnonzero(f & np.r_[True,~f[:-1] | (np.diff(x)!=1)])
    e=np.flatnonzero(f & np.r_[~f[1:] | (np.diff(x)!=1),True])
    assert len(s)==len(e)
    return s,e

def episodes(labels):
    y=np.asarray(labels,dtype=bool)
    return np.flatnonzero(y & np.r_[True,~y[:-1]]),np.flatnonzero(y & np.r_[~y[1:],True])

# Los 21 pares originales: cuatro métodos x cinco configuraciones + Persistencia FULL.
PAIRS=[(m,c) for m in METHODS if m!='PERSIST' for c in CONFIGS]+[('PERSIST','FULL')]
print('Pares de modelo/configuración:',len(PAIRS))

Pares de modelo/configuración: 21


In [3]:
# Lectura por combinación/archivo; cada resultado se computa directamente de predicciones.
predict_rows=[]; detect_rows=[]; epi_rows=[]; run_rows=[]; comp_rows=[]; sens_rows=[]; pair_checks=[]
for method,config in PAIRS:
    variables,mu,sd,tau=get_threshold(method,config)
    calibration_path=locate('predicciones',method,config,'calibration') if False else ROOT/'predicciones'/f'{method}_{config}_calibration.csv.gz'
    cal=load_csv(calibration_path,usecols=['error__'+v for v in variables]);cal_err=cal.to_numpy(dtype=np.float64)
    cal_scores=np.sqrt(np.mean(((cal_err-mu)/sd)**2,axis=1))
    quantiles={q:float(np.quantile(cal_scores,q)) for q in Q_VALUES}
    # Verificación del umbral original y reproducción de q=0.995.
    if not np.isclose(quantiles[0.995],tau,rtol=1e-7,atol=1e-7):
        raise ValueError(f'Umbral inconsistente: {method}/{config}: JSON={tau}, recalculado={quantiles[0.995]}')
    for test in TESTS:
        required=['indice','timestamp','Attack'] + [f'real__{v}' for v in variables] + [f'pred__{v}' for v in variables] + [f'error__{v}' for v in variables]
        pred_path=locate('predicciones',method,config,test)
        df=load_csv(pred_path,usecols=required)
        n=len(df)
        if n != EXPECTED_N[test]:raise ValueError(f'Cantidad inesperada {pred_path}: {n}')
        index=df.indice.to_numpy(dtype=np.int64)
        if not np.all(np.diff(index)==1):raise ValueError(f'Índices discontinuos: {pred_path}')
        y=df[['real__'+v for v in variables]].to_numpy(dtype=np.float64)
        yp=df[['pred__'+v for v in variables]].to_numpy(dtype=np.float64)
        # Usar el residuo ORIGINAL almacenado, no restar nuevamente real-pred.
        # Los valores real/pred y error fueron redondeados de modo independiente
        # al guardarse; en alertas cercanas al umbral, recomputar la resta
        # puede modificar la clasificación binaria.
        er=df[['error__'+v for v in variables]].to_numpy(dtype=np.float64)
        z=(er-mu)/sd
        score=np.sqrt(np.mean(z*z,axis=1))
        label=df.Attack.to_numpy(dtype=int)
        ep_s,ep_e=episodes(label)
        if len(ep_s)!=EXPECTED_EPISODES[test]:raise ValueError(f'Episodios inconsistentes {method}/{config}/{test}: {len(ep_s)}')
        for j,v in enumerate(variables):
            yy=y[:,j]; pp=yp[:,j]; diff=er[:,j]
            sse=float(np.dot(diff,diff)); centered=yy-yy.mean();sst=float(np.dot(centered,centered))
            predict_rows.append(dict(metodo=method,configuracion=config,archivo=test,variable=v,componente=v.split('_')[0],n=n,
                                     r2=(1-sse/sst if sst>0 else np.nan),rmse=float(np.sqrt(np.mean(diff*diff))),mae=float(np.mean(abs(diff)))))
        for q,threshold in quantiles.items():
            flags=score>threshold
            es=np.array([int(flags[a:b+1].any()) for a,b in zip(ep_s,ep_e)])
            sens_rows.append(dict(metodo=method,configuracion=config,archivo=test,cuantil=q,umbral=threshold,
                                  n=n,registros=int(flags.sum()),pds=100*flags.mean(),episodios=len(ep_s),coincidentes=int(es.sum())))
        flags=score>tau
        rs,re=runs(index,flags)
        lengths=index[re]-index[rs]+1
        detect_rows.append(dict(metodo=method,configuracion=config,archivo=test,n=n,registros=int(flags.sum()),pds=100*flags.mean(),
                                episodios=len(ep_s),coincidentes=int(sum(flags[a:b+1].any() for a,b in zip(ep_s,ep_e))),
                                intervalos=len(lengths),mediana_intervalo=float(np.median(lengths)) if len(lengths) else 0,
                                p90_intervalo=float(np.percentile(lengths,90)) if len(lengths) else 0,max_intervalo=int(max(lengths)) if len(lengths) else 0))
        for j,(a,b) in enumerate(zip(ep_s,ep_e),1):
            n_attack=int(flags[a:b+1].sum()); dur=b-a+1
            epi_rows.append(dict(metodo=method,configuracion=config,archivo=test,episodio=j,
                                 registros=dur,registros_senalados=n_attack,cobertura=100*n_attack/dur,coincidente=int(n_attack>0)))
        for j,(a,b) in enumerate(zip(rs,re),1):
            run_rows.append(dict(metodo=method,configuracion=config,archivo=test,intervalo=j,indice_inicio=int(index[a]),indice_fin=int(index[b]),longitud=int(index[b]-index[a]+1)))
        if flags.any():
            energy=(z[flags]**2).sum(axis=0)
            total=float(energy.sum())
            for comp in COMPS:
                share=float(energy[[v.startswith(comp+'_') for v in variables]].sum()/total*100) if total else 0
                comp_rows.append(dict(metodo=method,configuracion=config,archivo=test,componente=comp,participacion_error_pct=share))
        if test!='test1':
            dp=locate('deteccion',method,config,test)
            dd=load_csv(dp,usecols=['indice','alerta'])
            assert np.array_equal(index,dd.indice.to_numpy()), f'Índices distintos {dp}'
            original=dd.alerta.to_numpy(dtype=int)
            same=np.array_equal(flags.astype(int),original)
            mismatches=int(np.count_nonzero(flags.astype(int)!=original))
            pair_checks.append(dict(metodo=method,configuracion=config,archivo=test,alertas_reproducidas=same,discrepancias=mismatches))
            if not same:
                diff_idx=np.flatnonzero(flags.astype(int)!=original)
                ejemplo=[{'indice':int(index[k]),'score':float(score[k]),'umbral':float(tau),'nuevo':int(flags[k]),'original':int(original[k])} for k in diff_idx[:5]]
                raise ValueError(f'Detección original no reproducida: {method}/{config}/{test}. Discrepancias={mismatches}. Ejemplos={ejemplo}. Comprueba la versión de los archivos de predicción, residuos y umbral.')
    print('Consolidado',method,config)

prediction=pd.DataFrame(predict_rows); detection=pd.DataFrame(detect_rows)
episodes_df=pd.DataFrame(epi_rows); intervalos=pd.DataFrame(run_rows)
components=pd.DataFrame(comp_rows); sensitivity=pd.DataFrame(sens_rows)
checks=pd.DataFrame(pair_checks)
# Punto de control: persistir tablas intermedias sin tocar resultados originales.
CHECKPOINT = OUT/'_intermedios'
CHECKPOINT.mkdir(parents=True,exist_ok=True)
for nombre,tabla in {'prediction':prediction,'detection':detection,'episodes_df':episodes_df,
                     'intervalos':intervalos,'components':components,'sensitivity':sensitivity,
                     'checks':checks}.items():
    tabla.to_csv(CHECKPOINT/f'{nombre}.csv',index=False)
print('Punto de control creado:',CHECKPOINT)
assert len(prediction)==21*4*24 and len(detection)==21*4
assert checks.alertas_reproducidas.all()
print('Combinaciones señal/archivo:',len(prediction),'Verificaciones detectores previos:',len(checks))

Consolidado DT FULL
Consolidado DT SIN_P1
Consolidado DT SIN_P2
Consolidado DT SIN_P3
Consolidado DT SIN_P4
Consolidado RF FULL
Consolidado RF SIN_P1
Consolidado RF SIN_P2
Consolidado RF SIN_P3
Consolidado RF SIN_P4
Consolidado ET FULL
Consolidado ET SIN_P1
Consolidado ET SIN_P2
Consolidado ET SIN_P3
Consolidado ET SIN_P4
Consolidado XGB FULL
Consolidado XGB SIN_P1
Consolidado XGB SIN_P2
Consolidado XGB SIN_P3
Consolidado XGB SIN_P4
Consolidado PERSIST FULL
Punto de control creado: C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive\resultados_hai_arboles\analisis\consolidado_4_test\_intermedios
Combinaciones señal/archivo: 2016 Verificaciones detectores previos: 63


In [4]:
# Comprobación de dependencias antes de agregar resultados.
# Si el kernel se reinició después del cálculo anterior, recuperar el punto de control.
NECESARIAS = ('prediction','detection','episodes_df','intervalos','components','sensitivity','checks')
if any(nombre not in globals() for nombre in NECESARIAS):
    checkpoint = OUT/'_intermedios'
    archivos = {nombre:checkpoint/f'{nombre}.csv' for nombre in NECESARIAS}
    faltantes = [nombre for nombre,ruta in archivos.items() if not ruta.is_file()]
    if faltantes:
        raise RuntimeError('La celda de cálculo anterior (celda 4 de Jupyter) no ha terminado correctamente. '
                           'Ejecuta Kernel > Restart & Run All y verifica que aparezca '
                           '\"Punto de control creado\". No se calcularán tablas parciales. '
                           'Archivos intermedios ausentes: '+', '.join(faltantes))
    for nombre,ruta in archivos.items():
        globals()[nombre] = pd.read_csv(ruta)
    print('Se recuperaron las tablas intermedias desde:',checkpoint)
if len(prediction)!=21*4*24 or len(detection)!=21*4:
    raise RuntimeError('Los datos intermedios están incompletos; no se realizará la agregación.')

# Agregación correcta: medianas de 96 valores R² por método y configuración.
r2_by_test=prediction.groupby(['metodo','configuracion','archivo'],as_index=False).agg(r2_mediano=('r2','median'))
r2_global=prediction.groupby(['metodo','configuracion'],as_index=False).agg(r2_mediano_96=('r2','median'))
# Comparaciones pareadas sin promedio entre RMSE de señales diferentes.
full=prediction[prediction.configuracion=='FULL'].copy()
persist=full[full.metodo=='PERSIST'][['archivo','variable','rmse']].rename(columns={'rmse':'rmse_persistencia'})
compare=full[full.metodo!='PERSIST'].merge(persist,on=['archivo','variable'],validate='many_to_one')
compare['mejora_rmse']=compare.rmse < compare.rmse_persistencia
improvements=compare.groupby('metodo',as_index=False).agg(comparaciones=('mejora_rmse','size'),mejoras=('mejora_rmse','sum'))
# Diferencias pareadas de las propias señales excluidas.
piv=prediction[prediction.metodo!='PERSIST'].pivot(index=['metodo','archivo','variable','componente'],columns='configuracion',values='r2').reset_index()
abl_rows=[]
for comp in COMPS:
    sel=piv[piv.componente==comp]
    for method,g in sel.groupby('metodo'):
        d=g['SIN_'+comp]-g['FULL']
        abl_rows.append(dict(metodo=method,componente_excluido=comp,n=len(d),delta_r2_mediano=float(d.median())))
ablations=pd.DataFrame(abl_rows)
# Agregados de detección: sumar numeradores, no promediar porcentajes.
all_det=detection.groupby(['metodo','configuracion'],as_index=False).agg(registros_evaluados=('n','sum'),registros_senalados=('registros','sum'),episodios=('episodios','sum'),coincidentes=('coincidentes','sum'),intervalos=('intervalos','sum'))
all_det['pds']=100*all_det.registros_senalados/all_det.registros_evaluados
all_det['ce']=100*all_det.coincidentes/all_det.episodios
# La duración debe calcularse sobre los intervalos originales de los cuatro TEST.
continuity=intervalos.groupby(['metodo','configuracion'],as_index=False).agg(intervalos=('longitud','size'),mediana=('longitud','median'),p90=('longitud',lambda x:float(np.percentile(x,90))),maximo=('longitud','max'))
all_sens=sensitivity.groupby(['metodo','configuracion','cuantil'],as_index=False).agg(n=('n','sum'),registros=('registros','sum'),episodios=('episodios','sum'),coincidentes=('coincidentes','sum'))
all_sens['pds']=100*all_sens.registros/all_sens.n
all_sens['ce']=100*all_sens.coincidentes/all_sens.episodios
coverage=episodes_df[episodes_df.coincidente==1].groupby(['metodo','configuracion'],as_index=False).agg(cobertura_mediana_episodios_coincidentes=('cobertura','median'))
results={
'01_metricas_predictivas_4_test.csv':prediction,
'02_r2_por_test_4_test.csv':r2_by_test,
'03_r2_global_96_4_test.csv':r2_global,
'04_rmse_frente_persistencia_4_test.csv':improvements,
'05_deteccion_por_test_4_test.csv':detection,
'06_deteccion_global_4_test.csv':all_det,
'07_episodios_4_test.csv':episodes_df,
'08_intervalos_4_test.csv':intervalos,
'09_continuidad_4_test.csv':continuity,
'10_participacion_componentes_4_test.csv':components,
'11_ablaciones_r2_propio_4_test.csv':ablations,
'12_sensibilidad_por_test_4_test.csv':sensitivity,
'13_sensibilidad_global_4_test.csv':all_sens,
'14_cobertura_temporal_4_test.csv':coverage,
'15_verificacion_reproduccion_4_test.csv':checks,
}
for name,data in results.items():data.to_csv(OUT/name,index=False)
assert set(detection.archivo)==set(TESTS)
assert (all_det.episodios==58).all() and (all_det.registros_evaluados==361080).all()
print('Archivos:',len(results));display(all_det[all_det.configuracion=='FULL']);display(r2_global[r2_global.configuracion=='FULL']);display(improvements)

Archivos: 15


,metodo,configuracion,registros_evaluados,registros_senalados,episodios,coincidentes,intervalos,pds,ce
0,DT,FULL,361080,32877,58,44,4973,9.105184,75.862069
5,ET,FULL,361080,77588,58,50,10801,21.487759,86.206897
10,PERSIST,FULL,361080,2908,58,49,2301,0.805362,84.482759
11,RF,FULL,361080,70337,58,50,9872,19.479617,86.206897
16,XGB,FULL,361080,40146,58,47,4297,11.118312,81.034483


,metodo,configuracion,r2_mediano_96
0,DT,FULL,-0.020614
5,ET,FULL,0.391172
10,PERSIST,FULL,0.993351
11,RF,FULL,0.347950
16,XGB,FULL,0.964476


,metodo,comparaciones,mejoras
0,DT,96,11
1,ET,96,17
2,RF,96,16
3,XGB,96,39


In [5]:
# Figuras de resultados. Ambas incluyen siempre TEST1, TEST2, TEST3 y TEST4.
view=r2_by_test[r2_by_test.configuracion=='FULL'].pivot(index='archivo',columns='metodo',values='r2_mediano').reindex(TESTS)
fig,ax=plt.subplots(figsize=(9,5));view.plot(marker='o',ax=ax)
ax.set_xlabel('Conjunto de prueba');ax.set_ylabel('R² mediano de 24 señales');ax.set_title('Calidad predictiva FULL por conjunto (TEST1–TEST4)');ax.axhline(0,color='gray',linewidth=.8);fig.tight_layout()
fig.savefig(OUT/'fig_r2_4_test.png',dpi=180);plt.close(fig)
view=detection[detection.configuracion=='FULL'].pivot(index='archivo',columns='metodo',values='pds').reindex(TESTS)
fig,ax=plt.subplots(figsize=(9,5));view.plot(marker='o',ax=ax)
ax.set_xlabel('Conjunto de prueba');ax.set_ylabel('Registros señalados (%)');ax.set_title('PDS FULL en los cuatro TEST');fig.tight_layout()
fig.savefig(OUT/'fig_pds_4_test.png',dpi=180);plt.close(fig)
view=all_sens[all_sens.configuracion=='FULL'].pivot(index='cuantil',columns='metodo',values='pds')
fig,ax=plt.subplots(figsize=(9,5));view.plot(marker='o',ax=ax)
ax.set_xlabel('Cuantil de calibración');ax.set_ylabel('PDS (%)');ax.set_title('Sensibilidad del detector: TEST1–TEST4');fig.tight_layout()
fig.savefig(OUT/'fig_sensibilidad_pds_4_test.png',dpi=180);plt.close(fig)
view=all_sens[all_sens.configuracion=='FULL'].pivot(index='cuantil',columns='metodo',values='ce')
fig,ax=plt.subplots(figsize=(9,5));view.plot(marker='o',ax=ax)
ax.set_xlabel('Cuantil de calibración');ax.set_ylabel('Correspondencia de episodios (%)');ax.set_title('Sensibilidad: 58 episodios de TEST1–TEST4');fig.tight_layout()
fig.savefig(OUT/'fig_sensibilidad_ce_4_test.png',dpi=180);plt.close(fig)
print('Gráficas guardadas:',list(OUT.glob('fig_*.png')))

Gráficas guardadas: [WindowsPath('C:/Users/hugoc/Local/PUCV/Doctorado/Cursos/Semestre-2-2026/Industria-Inteligente/archive/resultados_hai_arboles/analisis/consolidado_4_test/fig_pds_4_test.png'), WindowsPath('C:/Users/hugoc/Local/PUCV/Doctorado/Cursos/Semestre-2-2026/Industria-Inteligente/archive/resultados_hai_arboles/analisis/consolidado_4_test/fig_r2_4_test.png'), WindowsPath('C:/Users/hugoc/Local/PUCV/Doctorado/Cursos/Semestre-2-2026/Industria-Inteligente/archive/resultados_hai_arboles/analisis/consolidado_4_test/fig_sensibilidad_ce_4_test.png'), WindowsPath('C:/Users/hugoc/Local/PUCV/Doctorado/Cursos/Semestre-2-2026/Industria-Inteligente/archive/resultados_hai_arboles/analisis/consolidado_4_test/fig_sensibilidad_pds_4_test.png')]


## Interpretación y controles pendientes

- Los resultados agregados se generan desde **cuatro TEST**, sin mezclar ventanas entre archivos.
- Las ablaciones comparan las **mismas 24 señales** y se calculan sobre 96 comparaciones por método/configuración.
- Los archivos de TEST1 son producidos por el notebook 11. Los tres TEST restantes se leen de los directorios originales.
- El contraste con etiquetas de ataque sirve solo como correspondencia externa, nunca como validación de falla física.
- Las gráficas de **Real vs. estimado** de episodios particulares siguen requiriendo selección de episodios; su actualización editorial debe hacerse usando los cuatro TEST, sin suponer que los episodios antiguos representan todo el conjunto.
- **No se ha ejecutado aquí este consolidado contra todos los modelos y predicciones**. Ejecutar todas las celdas y enviar los CSV de `analisis/consolidado_4_test` antes de actualizar cifras y figuras del artículo.